# CymbalGoal AlloyDB Workshop—SQL Server Package: build the legacy `CymbalGoalScouting`

**Track A, pipeline step 05 (Module 1, aldb001).** Builds the legacy SQL Server database students
migrate to AlloyDB in Module 1, exports it as a `.bak`, then restores that `.bak` exactly the way Start
Lab will and turns on change data capture, timing each step.

| | |
| :-- | :-- |
| **Runs in** | Colab Enterprise in the lab project, **after** `build/checks/module1/step1-provision.sh` has created the Cloud SQL for SQL Server instance |
| **Reads** | the eight staged tables at `gs://class-demo/alloydb-labs/cymbalgoal/` (**read-only**: another team's labs use it) and the T-SQL in `pipeline/sqlserver/` |
| **Writes** | the `.bak`, the scripts as they ran, a manifest and this run's handback under `gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop/sqlserver/` and `.../module1-checks/` |
| **Calls** | no Gemini, no Vertex. Cloud SQL Admin (export/import), Secret Manager (the `sqlserver` password Terraform generated) |

### The package

`CymbalGoalScouting`: the eight relational-core tables in `dbo` with 2014-era types (`NVARCHAR`,
`DATE`, `DATETIME2`, `MONEY`, `BIT`, `TINYINT`, `IDENTITY`), the data's real dirty edges kept, and a
legacy programmability layer: one scalar function, one view, one trigger and seven stored procedures
(cursors, temp tables, `TOP`, `ISNULL`, `+` concatenation, `FOR XML PATH`, `MERGE`, `TRY/CATCH`,
`OUTPUT` parameters).

**The deliberate conversion trap** is `usp_search_players`: the database collation is case-insensitive,
so `'messi'` finds Lionel Messi; PostgreSQL's `LIKE` is case-sensitive, so a faithful conversion
silently finds nothing. Section 6 records the SQL Server behaviour the conversion has to match.

### How to run

**Run all.** About 20–30 minutes, mostly the bulk load and the export/import. The SQL scripts are read
from `/content/sqlserver/` if you uploaded them there, otherwise from the repo on GitHub (`main`).
Every section ends with a **📋 SEND BACK** block, and the last cell prints them all together: paste
that one block back.

| | Section | What it does |
| :-- | :-- | :-- |
| 1 | Configuration | names, knobs, expected row counts |
| 2 | Environment and guards | project, the Cloud SQL instance, the write grant, the password |
| 3 | Scripts | fetch the T-SQL, split it into batches |
| 4 | Source data | read the staged tables, check counts, measure text widths |
| 5 | Build | create `CymbalGoalScouting_build`, bulk-load, validate keys, index, add programmability |
| 6 | Smoke tests | call every object; record the SQL Server answers the conversion must match |
| 7 | Export | `.bak` to the in-project staging bucket; then the write tests, rolled back |
| 8 | Restore as Start Lab will | drop the build copy, `gcloud sql import bak` as `CymbalGoalScouting`, verify |
| 9 | CDC | run `03-enable-cdc.sql`, check the capture jobs |
| 10 | Publish | `.bak`, scripts as run, manifest, smoke-test answers to the workshop bucket |
| 11 | Handback | every SEND BACK block in one place |

---
## 1 · Configuration

In [ ]:
SQL_INSTANCE     = "cymbalgoal-legacy-sql"            # build/checks/module1/terraform
DB_BUILD         = "CymbalGoalScouting_build"         # built, exported, then dropped
DB_FINAL         = "CymbalGoalScouting"               # restored from the .bak, as Start Lab will
SECRET_SQLSERVER = "cymbalgoal-sqlserver-password"    # the `sqlserver` login, written by Terraform

GCS_READ       = "gs://class-demo/alloydb-labs/cymbalgoal"                   # READ-ONLY
GCS_WORK       = "gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop"
PKG_PREFIX     = f"{GCS_WORK}/sqlserver"
RESULTS_PREFIX = f"{GCS_WORK}/module1-checks"
REPO_RAW       = "https://raw.githubusercontent.com/haggman/cymbalgoal-alloydb-workshop/main/pipeline/sqlserver"
BAK_NAME       = "CymbalGoalScouting.bak"

SEASON_FROM   = None   # e.g. 2019: keep games, appearances and events from that season on (smaller package)
REBUILD_FINAL = True   # drop CymbalGoalScouting (disabling CDC first) if an earlier run left one

BIG5 = ["GB1", "ES1", "IT1", "L1", "FR1"]
LOAD_ORDER_DEFAULT = ["competitions", "clubs", "players", "games",
                      "appearances", "game_events", "player_valuations", "transfers"]
EXPECTED_ROWS = {      # pipeline/00 (Part 2's staged counts); a mismatch is reported, not fatal
    "competitions": 65, "clubs": 796, "players": 13_439, "games": 29_740,
    "appearances": 832_193, "game_events": 417_617, "player_valuations": 297_822, "transfers": 65_494,
}

# Smoke-test anchors: Lionel Messi, FC Barcelona, the Premier League
SMOKE_PLAYER, SMOKE_CLUB, SMOKE_SEASON = 28003, 131, 2014
LEAGUE_CHECK = ("GB1", 2023)
LOAD_CHUNK = 100_000   # rows per bulk-copy call

---
## 2 · Environment and guards

Three things must be true before anything is built: the Cloud SQL instance from step 1 exists and is
running, this identity can write under `cymbalgoal-alloydb-workshop/` (the grant the Oct 14 runs were
missing), and the `sqlserver` password is in Secret Manager. Every bucket call goes through `gcs()`,
which refuses any write under the read-only `cymbalgoal/` prefix.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "cloud-sql-python-connector[pytds]", "pandas"], check=True)
print("client libraries ready")

In [ ]:
import csv, gzip, hashlib, io, json, os, re, time, urllib.request
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

HB, TIMERS, GCS_LOG = {}, {}, []
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
WORK = Path("/content/cymbalgoal-sqlserver")
try:
    WORK.mkdir(parents=True, exist_ok=True)
except OSError:
    WORK = Path.home() / "cymbalgoal-sqlserver"
    WORK.mkdir(parents=True, exist_ok=True)


def hb(section, line=""):
    HB.setdefault(section, []).append(str(line))


def send_back(section):
    print(f"\n📋 SEND BACK — {section}\n" + "-" * 78)
    for line in HB.get(section, []):
        print(line)
    print("-" * 78)


@contextmanager
def timed(name):
    t0 = time.time()
    try:
        yield
    finally:
        TIMERS[name] = time.time() - t0
        print(f"⏱  {name}: {TIMERS[name]:,.1f} s")


def sh(cmd, check=True, timeout=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}\n{(r.stderr or r.stdout).strip()[:2000]}")
    return r.stdout.strip() if check else r


def gcs(verb, *args, binary=False):
    """gcloud storage, with the read-only guard. Writes may only target the workshop prefix or this project's staging bucket."""
    uris = [a for a in args if str(a).startswith("gs://")]
    if verb in ("cp", "mv", "rm"):
        targets = uris[-1:] if verb != "rm" else uris
        for u in targets:
            if u.startswith(GCS_READ + "/") or u == GCS_READ:
                raise PermissionError(f"gcs(): refusing to write {u}: {GCS_READ}/ is read-only")
            if not (u.startswith(GCS_WORK + "/") or u.startswith(STAGING + "/")):
                raise PermissionError(f"gcs(): {u} is outside the writable prefixes")
    GCS_LOG.append("gcloud storage " + verb + " " + " ".join(map(str, args)))
    return subprocess.run(["gcloud", "storage", verb, *map(str, args)], capture_output=True, text=not binary)


def wait_sql_op(op_name, label):
    """Wait for a Cloud SQL operation without gcloud's 300-second default timeout."""
    r = sh(f"gcloud sql operations wait {op_name} --timeout=unlimited --format=json", check=False)
    if r.returncode != 0:
        raise RuntimeError(f"{label} failed: {(r.stderr or r.stdout).strip()[:1500]}")
    info = json.loads(sh(f"gcloud sql operations describe {op_name} --format=json"))
    if info.get("error"):
        raise RuntimeError(f"{label} failed: {json.dumps(info['error'])[:1500]}")
    return info


S = "2 · Environment"
PROJECT = sh("gcloud config get-value project")
ACCOUNT = sh("gcloud config get-value account")
STAGING = f"gs://{PROJECT}-cymbalgoal-staging"

inst = json.loads(sh(f"gcloud sql instances describe {SQL_INSTANCE} --format=json"))
CONNECTION_NAME = inst["connectionName"]
REGION = inst["region"]
print(f"project   {PROJECT}\naccount   {ACCOUNT}\ninstance  {CONNECTION_NAME}  "
      f"{inst['databaseVersion']}  {inst['settings']['tier']}  {inst['state']}")
assert inst["state"] == "RUNNABLE", f"{SQL_INSTANCE} is {inst['state']}: wait for step 1 to finish"

# The write grant: fail now, not after 20 minutes of building.
_marker = WORK / "step2-started.txt"
_marker.write_text(f"step 2 (05-sqlserver-package) started {RUN_ID} by {ACCOUNT} in {PROJECT}\n")
r = gcs("cp", _marker, f"{RESULTS_PREFIX}/step2-started-{RUN_ID}.txt")
if r.returncode != 0:
    raise PermissionError(
        f"Cannot write under {GCS_WORK}/ as {ACCOUNT}.\n{r.stderr.strip()[:500]}\n\n"
        "Grant this lab account write access to the workshop prefix in class-demo, then re-run from here.")
print(f"write grant OK ({RESULTS_PREFIX}/)")

SQL_PASSWORD = sh(f"gcloud secrets versions access latest --secret={SECRET_SQLSERVER}")
print(f"sqlserver password read from Secret Manager ({len(SQL_PASSWORD)} chars)")

hb(S, f"project {PROJECT}  account {ACCOUNT}  run {RUN_ID}")
hb(S, f"instance {CONNECTION_NAME}: {inst['databaseVersion']}, {inst['settings']['tier']}, "
      f"edition {inst['settings'].get('edition', '?')}, {inst['state']}")
hb(S, "write grant OK; password from Secret Manager OK")
send_back(S)

---
## 3 · Scripts

`01-schema.sql` (tables, then after the `@@AFTER_LOAD` marker the key validation and indexes),
`02-programmability.sql` (function, view, trigger, procedures) and `03-enable-cdc.sql`. They are split
on `GO` lines into batches, the way `sqlcmd` and SSMS send them.

In [ ]:
S = "3 · Scripts"
SCRIPT_NAMES = ["01-schema.sql", "02-programmability.sql", "03-enable-cdc.sql"]


def load_scripts():
    for d in [Path("/content/sqlserver"), Path.cwd() / "sqlserver", Path.cwd() / "pipeline" / "sqlserver",
              Path.cwd().parent / "pipeline" / "sqlserver"]:
        if all((d / n).exists() for n in SCRIPT_NAMES):
            return {n: (d / n).read_text(encoding="utf-8") for n in SCRIPT_NAMES}, str(d)
    out = {}
    for n in SCRIPT_NAMES:
        try:
            with urllib.request.urlopen(f"{REPO_RAW}/{n}", timeout=30) as resp:
                out[n] = resp.read().decode("utf-8")
        except Exception as e:  # noqa: BLE001
            raise RuntimeError(
                f"Could not fetch {REPO_RAW}/{n} ({e}).\nPush the repo, or upload the three files in "
                f"pipeline/sqlserver/ to /content/sqlserver/ with the file browser, then re-run this cell.")
    return out, REPO_RAW


def split_batches(sql):
    """Split on GO lines; drop batches that are only comments or blank."""
    batches, cur = [], []
    for line in sql.splitlines() + ["GO"]:
        if re.fullmatch(r"\s*GO\s*", line, flags=re.I):
            body = "\n".join(cur).strip()
            if any(l.strip() and not l.strip().startswith("--") for l in body.splitlines()):
                batches.append(body)
            cur = []
        else:
            cur.append(line)
    return batches


def batch_label(b):
    m = re.search(r"CREATE\s+(TABLE|VIEW|FUNCTION|PROCEDURE|TRIGGER|NONCLUSTERED INDEX)\s+([\w.]+)", b, flags=re.I)
    if m:
        return f"{m.group(1).upper()} {m.group(2)}"
    first = next(l for l in b.splitlines() if l.strip() and not l.strip().startswith("--"))
    return first.strip()[:70]


SCRIPTS, SCRIPTS_FROM = load_scripts()
for n, t in SCRIPTS.items():
    print(f"  {n:<26}{len(t):>8,} chars  sha256 {hashlib.sha256(t.encode()).hexdigest()[:12]}  "
          f"{len(split_batches(t))} batches")
assert "-- @@AFTER_LOAD" in SCRIPTS["01-schema.sql"], "01-schema.sql has no @@AFTER_LOAD marker"
hb(S, f"scripts from {SCRIPTS_FROM}")
for n, t in SCRIPTS.items():
    hb(S, f"  {n}: sha256 {hashlib.sha256(t.encode()).hexdigest()[:12]}, {len(split_batches(t))} batches")
send_back(S)

---
## 4 · Source data

The staged tables are headerless `.csv.gz` files whose column order is in the bucket's
`manifest.json` (or, failing that, the staged `schema.sql`). Everything is read as text: SQL Server
converts each value to the column's type during the bulk load, the same way it would from a file.

Two measurements decide the final DDL: the longest value in every `NVARCHAR` column (a declared width
too small is widened to the next standard size, and the change is reported), and whether
`game_events.game_event_id` is upstream's text key or the pipeline's numeric surrogate.

In [ ]:
S = "4 · Source data"


def read_text_object(uri):
    r = gcs("cat", uri)
    if r.returncode != 0:
        raise RuntimeError(f"cannot read {uri}: {r.stderr.strip()[:300]}")
    return r.stdout


def columns_from_ddl(sql):
    """Part 1's staged schema.sql: column order per CREATE TABLE, profile columns excluded."""
    out = {}
    for m in re.finditer(r"CREATE TABLE (\w+) \((.*?)\n\);", sql, flags=re.S):
        cols = []
        for line in m.group(2).splitlines():
            mm = re.match(r"\s+(\w+)\s+[A-Z{]", line)
            if mm and mm.group(1).upper() not in ("PRIMARY", "FOREIGN", "CONSTRAINT", "UNIQUE"):
                if mm.group(1) not in ("profile_text", "profile_embedding"):
                    cols.append(mm.group(1))
        out[m.group(1)] = cols
    return out


with timed("read manifest"):
    try:
        MF = json.loads(read_text_object(f"{GCS_READ}/manifest.json"))
        MF_SRC = "manifest.json"
    except Exception as e:  # noqa: BLE001
        print(f"⚠️  manifest.json unreadable ({e}); using schema.sql column order")
        MF, MF_SRC = {}, "schema.sql"
STAGED = MF.get("staged_files", {})
LOAD_ORDER = MF.get("load_order") or LOAD_ORDER_DEFAULT
DDL_COLS = None
SRC_COLS = {}
for t in LOAD_ORDER:
    if t in STAGED and STAGED[t].get("column_order"):
        SRC_COLS[t] = STAGED[t]["column_order"]
    else:
        if DDL_COLS is None:
            DDL_COLS = columns_from_ddl(read_text_object(f"{GCS_READ}/schema.sql"))
        SRC_COLS[t] = DDL_COLS[t]
        MF_SRC = "schema.sql (some tables)"


def read_table(t):
    r = gcs("cat", f"{GCS_READ}/{t}.csv.gz", binary=True)
    if r.returncode != 0:
        raise RuntimeError(f"cannot read {t}.csv.gz: {r.stderr.decode(errors='replace')[:300]}")
    raw = r.stdout
    data = gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw   # gcloud may decompress for us
    first = next(csv.reader(io.StringIO(data[:200_000].decode("utf-8", errors="replace"))))
    if len(first) != len(SRC_COLS[t]):
        raise RuntimeError(f"{t}: file has {len(first)} fields, column list has {len(SRC_COLS[t])}")
    return pd.read_csv(io.BytesIO(data), header=None, names=SRC_COLS[t], dtype=str,
                       keep_default_na=False, na_values=[""], low_memory=False), len(raw)


SRC, SRC_BYTES = {}, {}
with timed("read source tables"):
    for t in LOAD_ORDER:
        SRC[t], SRC_BYTES[t] = read_table(t)
        exp = EXPECTED_ROWS.get(t)
        flag = "" if exp in (None, len(SRC[t])) else f"   ⚠️ expected {exp:,}"
        print(f"  {t:<20}{len(SRC[t]):>10,} rows {len(SRC[t].columns):>3} cols  {SRC_BYTES[t] / 1e6:>6.1f} MB gz{flag}")

if SEASON_FROM:
    keep = set(SRC["games"].loc[SRC["games"].season.astype(int) >= SEASON_FROM, "game_id"])
    for t in ("games", "appearances", "game_events"):
        before = len(SRC[t])
        SRC[t] = SRC[t][SRC[t].game_id.isin(keep)].copy()
        print(f"  SEASON_FROM {SEASON_FROM}: {t} {before:,} -> {len(SRC[t]):,}")

LATEST_SEASON = int(SRC["games"].season.astype(int).max())
_ev = SRC["game_events"].game_event_id
EVENT_PK_NUMERIC = bool(_ev.str.fullmatch(r"\d{1,18}").all())
print(f"\nlatest season {LATEST_SEASON}; game_event_id is "
      f"{'numeric (BIGINT)' if EVENT_PK_NUMERIC else 'text (NVARCHAR)'}")

hb(S, f"column order from {MF_SRC}; load order {', '.join(LOAD_ORDER)}")
hb(S, "rows: " + ", ".join(f"{t} {len(SRC[t]):,}" for t in LOAD_ORDER))
_mis = [t for t in LOAD_ORDER if EXPECTED_ROWS.get(t) not in (None, len(SRC[t]))]
hb(S, "row counts vs EXPECTED_ROWS: " + ("all match" if not _mis else "MISMATCH " + ", ".join(_mis))
   + (f" (SEASON_FROM={SEASON_FROM})" if SEASON_FROM else ""))
hb(S, f"game_event_id: {'BIGINT' if EVENT_PK_NUMERIC else 'NVARCHAR'}; latest season {LATEST_SEASON}")
send_back(S)

In [ ]:
# ---- Map the source onto the SQL Server tables -----------------------------------------------
S = "4 · Source data"
INT_TYPES = {"INT": (-2**31, 2**31 - 1), "BIGINT": (-2**63, 2**63 - 1),
             "SMALLINT": (-32768, 32767), "TINYINT": (0, 255)}
COL_RE = re.compile(r"^\s{4}(\w+)\s+(NVARCHAR\((?:\d+|MAX)\)|BIGINT|INT|SMALLINT|TINYINT|DATE|DATETIME2\(\d\)|"
                    r"MONEY|DECIMAL\(\d+,\d+\)|BIT)\s*(IDENTITY\(\d+,\d+\))?\s*(NOT NULL|NULL)(.*)$")
WIDTHS = [10, 20, 30, 40, 60, 100, 200, 400, 1000, 2000, 4000]


def parse_schema(sql):
    tables, cur = {}, None
    for line in sql.splitlines():
        m = re.match(r"\s*CREATE TABLE dbo\.(\w+)\s*\(", line)
        if m:
            cur = m.group(1); tables[cur] = []; continue
        if cur and re.match(r"\s*\);", line):
            cur = None; continue
        if cur:
            m = COL_RE.match(line)
            if m:
                tables[cur].append({"name": m.group(1), "type": m.group(2), "identity": bool(m.group(3)),
                                    "nullable": m.group(4) == "NULL", "default": "DEFAULT" in m.group(5)})
    return tables


def set_col_type(sql, table, col, new_type):
    start = sql.index(f"CREATE TABLE dbo.{table} (")
    end = sql.index("\n);", start)
    block, n = re.subn(rf"(?m)^(\s{{4}}{col}\s+)\S+", rf"\g<1>{new_type}", sql[start:end], count=1)
    assert n == 1, f"could not retype {table}.{col}"
    return sql[:start] + block + sql[end:]


DERIVED = {
    "clubs":   {"is_big5":   lambda d: d.domestic_competition_id.isin(BIG5).map({True: "1", False: "0"})},
    "players": {"is_active": lambda d: (pd.to_numeric(d.last_season, errors="coerce").fillna(0) >= LATEST_SEASON)
                                       .map({True: "1", False: "0"})},
}
SORT_KEYS = {"transfers": ["transfer_date", "player_id", "transfer_id"], "player_valuations": ["player_id", "valuation_date"]}

SCHEMA_SQL = SCRIPTS["01-schema.sql"]
if EVENT_PK_NUMERIC:
    SCHEMA_SQL = set_col_type(SCHEMA_SQL, "game_events", "game_event_id", "BIGINT")
SCHEMA = parse_schema(SCHEMA_SQL)
assert set(SCHEMA) == set(LOAD_ORDER), f"schema tables {sorted(SCHEMA)} != source tables {sorted(LOAD_ORDER)}"


def utf16_len(s):
    return len(s.encode("utf-16-le")) // 2


LOAD, WIDENED, DROPPED, NOTES = {}, [], {}, []
for t in LOAD_ORDER:
    df = SRC[t]
    for k, fn in DERIVED.get(t, {}).items():
        df[k] = fn(df)
    if t in SORT_KEYS:   # IDENTITY values follow load order, so make it deterministic
        keys = {k: (pd.to_numeric(df[k], errors="coerce") if k.endswith("_id") else df[k]) for k in SORT_KEYS[t]}
        order = pd.DataFrame(keys).sort_values(list(keys), kind="mergesort").index
        df = df.loc[order]
    cols = []
    for c in SCHEMA[t]:
        if c["identity"]:
            continue
        if c["name"] in df.columns:
            cols.append(c)
        elif not (c["nullable"] or c["default"]):
            raise RuntimeError(f"{t}.{c['name']} is NOT NULL with no default and is not in the source")
    DROPPED[t] = [c for c in df.columns if c not in {x["name"] for x in cols}]
    out = pd.DataFrame(index=df.index)
    for c in cols:
        s, typ, name = df[c["name"]], c["type"], f"{t}.{c['name']}"
        if typ in INT_TYPES:
            s = s.str.replace(r"\.0+$", "", regex=True)
            bad = s.notna() & ~s.fillna("0").str.fullmatch(r"-?\d+")
            assert not bad.any(), f"{name}: non-integer values, e.g. {s[bad].head(3).tolist()}"
            lo, hi = INT_TYPES[typ]
            v = pd.to_numeric(s, errors="coerce")
            assert v.dropna().between(lo, hi).all(), f"{name}: values outside {typ} ({v.min()}..{v.max()})"
        elif typ == "DATE":
            s = s.str.slice(0, 10)
            bad = s.notna() & ~s.fillna("2000-01-01").str.fullmatch(r"\d{4}-\d{2}-\d{2}")
            assert not bad.any(), f"{name}: bad dates, e.g. {s[bad].head(3).tolist()}"
        elif typ == "MONEY" or typ.startswith("DECIMAL"):
            bad = s.notna() & ~s.fillna("0").str.fullmatch(r"-?\d+(\.\d+)?")
            assert not bad.any(), f"{name}: non-numeric values, e.g. {s[bad].head(3).tolist()}"
        elif typ.startswith("NVARCHAR") and typ != "NVARCHAR(MAX)":
            declared = int(re.search(r"\d+", typ).group())
            longest = int(s.dropna().map(utf16_len).max()) if s.notna().any() else 0
            if longest > declared:
                new = next((w for w in WIDTHS if w >= longest), None)
                new_type = f"NVARCHAR({new})" if new else "NVARCHAR(MAX)"
                SCHEMA_SQL = set_col_type(SCHEMA_SQL, t, c["name"], new_type)
                WIDENED.append(f"{name} {typ} -> {new_type} (longest {longest})")
        if not c["nullable"]:   # keep_nulls: a NULL would not fall back to the column default
            assert s.notna().all(), f"{name} is NOT NULL but has {int(s.isna().sum())} missing values"
        out[c["name"]] = s
    LOAD[t] = (list(out.columns), out)

SCHEMA = parse_schema(SCHEMA_SQL)
print("columns loaded per table (identity and default-only columns are filled by SQL Server):")
for t in LOAD_ORDER:
    print(f"  {t:<20}{len(LOAD[t][0]):>3} cols   dropped from source: {', '.join(DROPPED[t]) or '-'}")
print("\nNVARCHAR widths changed:" if WIDENED else "\nevery NVARCHAR width fits the data")
for w in WIDENED:
    print("  " + w)

hb(S, "dropped source columns: " + ("; ".join(f"{t}: {', '.join(v)}" for t, v in DROPPED.items() if v) or "none"))
hb(S, "NVARCHAR widths widened: " + ("; ".join(WIDENED) if WIDENED else "none"))
send_back(S)

---
## 5 · Build

Connects through the **Cloud SQL Python Connector** (`pytds` driver, public IP, no authorized network
needed) as the `sqlserver` login. Creates `CymbalGoalScouting_build`, runs the table half of
`01-schema.sql`, bulk-loads each table with TDS bulk copy (falling back to batched `INSERT` if bulk copy
fails), then runs the `@@AFTER_LOAD` half: every foreign key is re-validated (the bulk load skips
constraint checks) and the indexes are built. Last, `02-programmability.sql`.

Any batch that fails is reported by name and the run continues, so one bad procedure doesn't hide the
rest.

In [ ]:
from google.cloud.sql.connector import Connector
import pytds

S = "5 · Build"
connector = Connector(quota_project=PROJECT, refresh_strategy="lazy")


def connect(db="master", autocommit=True):
    return connector.connect(CONNECTION_NAME, "pytds", user="sqlserver", password=SQL_PASSWORD,
                             db=db, autocommit=autocommit)


def run_batches(conn, batches, label):
    ok, failed = 0, []
    cur = conn.cursor()
    for b in batches:
        try:
            cur.execute(b)
            while cur.nextset():
                pass
            ok += 1
        except Exception as e:  # noqa: BLE001
            failed.append(f"{batch_label(b)}: {type(e).__name__}: {str(e).strip()[:300]}")
    print(f"{label}: {ok} batches OK, {len(failed)} failed")
    for f in failed:
        print("  ❌ " + f)
    return ok, failed


def scalar(conn, sql, *params):
    cur = conn.cursor()
    cur.execute(sql, params or None)
    row = cur.fetchone()
    return row[0] if row else None


def drop_database(conn, name, cdc_first=False):
    if scalar(conn, "SELECT DB_ID(%s)", name) is None:
        return False
    cur = conn.cursor()
    if cdc_first and scalar(conn, "SELECT is_cdc_enabled FROM sys.databases WHERE name = %s", name):
        cur.execute(f"EXEC msdb.dbo.gcloudsql_cdc_disable_db '{name}'")
    try:
        cur.execute(f"ALTER DATABASE [{name}] SET SINGLE_USER WITH ROLLBACK IMMEDIATE")
    except Exception as e:  # noqa: BLE001
        print(f"  (SINGLE_USER refused: {str(e)[:120]}; dropping anyway)")
    cur.execute(f"DROP DATABASE [{name}]")
    return True


with timed("connect"):
    master = connect()
VERSION = scalar(master, "SELECT @@VERSION").splitlines()[0]
INSTANCE_COLLATION = scalar(master, "SELECT CONVERT(NVARCHAR(128), SERVERPROPERTY('Collation'))")
print(f"{VERSION}\ninstance collation {INSTANCE_COLLATION}")

if drop_database(master, DB_BUILD):
    print(f"dropped an earlier {DB_BUILD}")
if scalar(master, "SELECT DB_ID(%s)", DB_FINAL) is not None:
    if REBUILD_FINAL:
        drop_database(master, DB_FINAL, cdc_first=True)
        print(f"dropped an earlier {DB_FINAL} (REBUILD_FINAL)")
    else:
        raise RuntimeError(f"{DB_FINAL} exists; set REBUILD_FINAL = True to replace it")

with timed("create database"):
    master.cursor().execute(f"CREATE DATABASE [{DB_BUILD}]")
DB_COLLATION = scalar(master, "SELECT CONVERT(NVARCHAR(128), DATABASEPROPERTYEX(%s, 'Collation'))", DB_BUILD)
print(f"{DB_BUILD} collation {DB_COLLATION}")
assert "_CI_" in DB_COLLATION, f"collation {DB_COLLATION} is not case-insensitive: the deliberate trap needs CI"

PRE, POST = SCHEMA_SQL.split("-- @@AFTER_LOAD", 1)
db = connect(DB_BUILD)
with timed("create tables"):
    _, FAILED_PRE = run_batches(db, split_batches(PRE), "tables")
assert not FAILED_PRE, "table creation failed: fix 01-schema.sql before going on"

hb(S, VERSION)
hb(S, f"collation: instance {INSTANCE_COLLATION}, database {DB_COLLATION}")

In [ ]:
S = "5 · Build"
LOAD_STATS = {}


def rows_of(frame):
    return frame.astype(object).where(frame.notna(), None).itertuples(index=False, name=None)


def bulk_copy(conn, table, cols, frame):
    cur = conn.cursor()
    for i in range(0, len(frame), LOAD_CHUNK):
        part = frame.iloc[i:i + LOAD_CHUNK]
        cur.copy_to(table_or_view=table, schema="dbo", columns=cols, data=rows_of(part),
                    keep_nulls=True, tablock=True, check_constraints=False)


def insert_batches(conn, table, cols, frame):
    per = max(1, min(1000, 2000 // len(cols)))           # 2,100-parameter limit per statement
    cur = conn.cursor()
    collist = ", ".join(cols)
    for i in range(0, len(frame), per):
        part = list(rows_of(frame.iloc[i:i + per]))
        values = ", ".join("(" + ", ".join(["%s"] * len(cols)) + ")" for _ in part)
        cur.execute(f"INSERT INTO dbo.{table} ({collist}) VALUES {values}", [v for r in part for v in r])


with timed("bulk load (all tables)"):
    for t in LOAD_ORDER:
        cols, frame = LOAD[t]
        t0, method = time.time(), "bulk copy"
        try:
            bulk_copy(db, t, cols, frame)
        except Exception as e:  # noqa: BLE001
            print(f"  ⚠️  {t}: bulk copy failed ({type(e).__name__}: {str(e)[:200]}); falling back to INSERT")
            db.cursor().execute(f"DELETE FROM dbo.{t}")
            method = "batched INSERT"
            insert_batches(db, t, cols, frame)
        secs = time.time() - t0
        n = scalar(db, f"SELECT COUNT_BIG(*) FROM dbo.{t}")
        LOAD_STATS[t] = {"rows": int(n), "seconds": round(secs, 1), "method": method}
        flag = "" if n == len(frame) else f"   ❌ expected {len(frame):,}"
        print(f"  {t:<20}{n:>10,} rows  {secs:>7.1f} s  {n / max(secs, 0.01):>9,.0f} rows/s  {method}{flag}")
        assert n == len(frame), f"{t}: loaded {n:,} of {len(frame):,}"

with timed("validate keys and build indexes"):
    _, FAILED_POST = run_batches(db, split_batches(POST), "after load")

with timed("programmability"):
    PROG_BATCHES = split_batches(SCRIPTS["02-programmability.sql"])
    _, FAILED_PROG = run_batches(db, PROG_BATCHES, "programmability")

UNTRUSTED = scalar(db, "SELECT COUNT(*) FROM sys.foreign_keys WHERE is_not_trusted = 1")
OBJECTS = []
cur = db.cursor()
cur.execute("""SELECT o.type_desc, o.name FROM sys.objects o
               WHERE o.schema_id = SCHEMA_ID('dbo') AND o.is_ms_shipped = 0
                 AND o.type IN ('U','V','P','FN','TR')
               ORDER BY CASE o.type WHEN 'U' THEN 1 WHEN 'V' THEN 2 WHEN 'FN' THEN 3 WHEN 'TR' THEN 4 ELSE 5 END, o.name""")
OBJECTS = [(r[0], r[1]) for r in cur.fetchall()]
SIZE_MB = scalar(db, "SELECT CAST(SUM(size) * 8 / 1024.0 AS DECIMAL(10,1)) FROM sys.database_files")

for t in LOAD_ORDER:
    s = LOAD_STATS[t]
    hb(S, f"{t}: {s['rows']:,} rows in {s['seconds']} s ({s['method']})")
hb(S, f"after-load batches failed: {len(FAILED_POST)}; untrusted foreign keys: {UNTRUSTED}")
for f in FAILED_POST:
    hb(S, "  ❌ " + f)
hb(S, f"programmability: {len(PROG_BATCHES) - len(FAILED_PROG)}/{len(PROG_BATCHES)} batches OK")
for f in FAILED_PROG:
    hb(S, "  ❌ " + f)
hb(S, "objects: " + "; ".join(f"{k} {n}" for k, n in [(td, sum(1 for x in OBJECTS if x[0] == td))
                                                     for td in dict.fromkeys(o[0] for o in OBJECTS)]))
hb(S, "  " + ", ".join(n for td, n in OBJECTS if td != "USER_TABLE"))
hb(S, f"database size {SIZE_MB} MB")
send_back(S)

---
## 6 · Smoke tests

Every function, view and procedure is called with fixed arguments and its answer recorded. These are
the **SQL Server answers the converted AlloyDB code must reproduce**; step 3 runs the same calls against
the migrated database and compares. The key ones:

- `usp_search_players 'messi'` and `'MESSI'` must both find Lionel Messi (the case-insensitivity trap).
- `fn_player_age` must not count a birthday that hasn't happened yet.

In [ ]:
S = "6 · Smoke tests"
SMOKE_CALLS = {
    "search_lower":   "EXEC dbo.usp_search_players @name_pattern = N'messi';",
    "search_upper":   "EXEC dbo.usp_search_players @name_pattern = N'MESSI';",
    "search_mixed":   "EXEC dbo.usp_search_players @name_pattern = N'Messi';",
    "season_stats":   f"DECLARE @g INT, @m INT; EXEC dbo.usp_get_player_season_stats {SMOKE_PLAYER}, {SMOKE_SEASON}, @g OUTPUT, @m OUTPUT; "
                      "SELECT @g AS total_goals, @m AS total_minutes;",
    "league_table":   f"EXEC dbo.usp_club_season_table N'{LEAGUE_CHECK[0]}', {LEAGUE_CHECK[1]};",
    "top_scorers":    f"EXEC dbo.usp_top_scorers N'{LEAGUE_CHECK[0]}', {LEAGUE_CHECK[1]}, 10;",
    "career_path":    f"DECLARE @p NVARCHAR(4000); EXEC dbo.usp_player_career_path {SMOKE_PLAYER}, @p OUTPUT; SELECT @p AS career_path_out;",
    "profile_view":   f"SELECT * FROM dbo.vw_player_profile WHERE player_id = {SMOKE_PLAYER};",
    "age_before_bday": "SELECT dbo.fn_player_age('1987-06-24', '2026-06-23') AS age;",
    "age_on_bday":     "SELECT dbo.fn_player_age('1987-06-24', '2026-06-24') AS age;",
}


def all_results(cur):
    out = []
    while True:
        if cur.description:
            cols = [d[0] for d in cur.description]
            out.append({"columns": cols, "rows": [[None if v is None else str(v) for v in r] for r in cur.fetchall()]})
        if not cur.nextset():
            break
    return out


def call(conn, sql):
    cur = conn.cursor()
    try:
        cur.execute(sql)
        return {"ok": True, "results": all_results(cur)}
    except Exception as e:  # noqa: BLE001
        return {"ok": False, "error": f"{type(e).__name__}: {str(e).strip()[:300]}"}


SMOKE = {k: call(db, v) for k, v in SMOKE_CALLS.items()}


def first_rows(k):
    r = SMOKE[k]
    return r["results"][0]["rows"] if r["ok"] and r["results"] else []


def show(k, n=3):
    r = SMOKE[k]
    if not r["ok"]:
        return f"ERROR {r['error']}"
    sets = r["results"]
    return " | ".join(f"{len(s['rows'])} rows" + (f" first={s['rows'][0][:4]}" if s["rows"] else "") for s in sets[:2])


CHECKS = {
    "search 'messi' finds Lionel Messi": any(str(row[1]).startswith("Lionel Messi") for row in first_rows("search_lower")),
    "search is case-insensitive": len(first_rows("search_lower")) == len(first_rows("search_upper")) == len(first_rows("search_mixed")) > 0,
    "season stats return rows and totals": SMOKE["season_stats"]["ok"] and len(SMOKE["season_stats"]["results"]) == 2
                                           and len(SMOKE["season_stats"]["results"][0]["rows"]) > 0,
    f"league table {LEAGUE_CHECK} has 18-20 clubs": 18 <= len(first_rows("league_table")) <= 20,
    "top scorers returns >= 10 rows": len(first_rows("top_scorers")) >= 10,
    "career path is built": SMOKE["career_path"]["ok"] and any(
        s["rows"] and s["rows"][0][-1] for s in SMOKE["career_path"]["results"]),
    "view returns the player": len(first_rows("profile_view")) == 1,
    "fn_player_age: 38 the day before the birthday": first_rows("age_before_bday") == [["38"]],
    "fn_player_age: 39 on the birthday": first_rows("age_on_bday") == [["39"]],
}
for k in SMOKE_CALLS:
    print(f"  {k:<16} {show(k)}")
print()
for name, ok in CHECKS.items():
    print(("  ✅ " if ok else "  ❌ ") + name)
    hb(S, ("PASS " if ok else "FAIL ") + name)
for k in ("search_lower", "season_stats", "league_table", "career_path"):
    hb(S, f"  {k}: {show(k)}")
(WORK / "smoke-sqlserver.json").write_text(json.dumps({"calls": SMOKE_CALLS, "answers": SMOKE}, indent=1))
send_back(S)

---
## 7 · Export

`gcloud sql export bak` writes a full backup of the build database to this project's staging bucket.
It reads and writes **as the instance's service account**, which Terraform granted `objectAdmin` on that
bucket; it can't be granted anything on `class-demo`, so the copy to the workshop prefix happens in
section 10.

After the export, the write paths are tested against the build copy **inside a transaction that is
rolled back**: `usp_upsert_valuation` (MERGE: an INSERT then an UPDATE), `usp_record_transfer`
(nested transaction, `SCOPE_IDENTITY()`), and the `modified_at` trigger. None of it reaches the `.bak`.

In [ ]:
S = "7 · Export"
BAK_STAGING = f"{STAGING}/sqlserver/{BAK_NAME}"

db.close()
with timed("export .bak"):
    op = sh(f"gcloud sql export bak {SQL_INSTANCE} {BAK_STAGING} --database={DB_BUILD} --async --quiet --format='value(name)'")
    wait_sql_op(op, "export")
BAK_META = json.loads(sh(f"gcloud storage objects describe {BAK_STAGING} --format=json"))
BAK_BYTES = int(BAK_META.get("size", 0))
print(f"{BAK_STAGING}  {BAK_BYTES / 1e6:,.1f} MB")

# ---- write paths, rolled back ----------------------------------------------------------------
tx = connect(DB_BUILD, autocommit=False)
WRITE_TESTS = {}
try:
    cur = tx.cursor()
    cur.execute(f"DECLARE @a NVARCHAR(10); EXEC dbo.usp_upsert_valuation {SMOKE_PLAYER}, '2099-01-01', 12345678, NULL, @a OUTPUT; SELECT @a;")
    WRITE_TESTS["upsert #1 action"] = all_results(cur)[-1]["rows"][0][0]
    cur.execute(f"DECLARE @a NVARCHAR(10); EXEC dbo.usp_upsert_valuation {SMOKE_PLAYER}, '2099-01-01', 23456789, NULL, @a OUTPUT; SELECT @a;")
    WRITE_TESTS["upsert #2 action"] = all_results(cur)[-1]["rows"][0][0]
    cur.execute(f"SELECT CONVERT(NVARCHAR(30), market_value_in_eur), CONVERT(NVARCHAR(30), modified_at, 126) "
                f"FROM dbo.players WHERE player_id = {SMOKE_PLAYER}")
    mv, mod = cur.fetchone()
    WRITE_TESTS["players.market_value after upsert"] = mv
    WRITE_TESTS["trigger stamped modified_at"] = mod is not None
    max_id = scalar(tx, "SELECT MAX(transfer_id) FROM dbo.transfers")
    cur.execute(f"DECLARE @id INT; EXEC dbo.usp_record_transfer {SMOKE_PLAYER}, {SMOKE_CLUB}, '2099-07-01', 0, @id OUTPUT; SELECT @id;")
    new_id = all_results(cur)[-1]["rows"][0][0]
    WRITE_TESTS["record_transfer id"] = f"{new_id} (max before {max_id})"
    WRITE_TESTS["player moved"] = scalar(tx, f"SELECT current_club_id FROM dbo.players WHERE player_id = {SMOKE_PLAYER}") == SMOKE_CLUB
    try:
        cur.execute("DECLARE @a NVARCHAR(10); EXEC dbo.usp_upsert_valuation -1, '2099-01-01', 1, NULL, @a OUTPUT;")
        WRITE_TESTS["unknown player raises"] = False
    except Exception as e:  # noqa: BLE001
        WRITE_TESTS["unknown player raises"] = "does not exist" in str(e)
finally:
    tx.rollback()
    tx.close()
EXPECT_WRITES = {"upsert #1 action": "INSERT", "upsert #2 action": "UPDATE",
                 "players.market_value after upsert": "23456789.00", "trigger stamped modified_at": True,
                 "player moved": True, "unknown player raises": True}
for k, v in WRITE_TESTS.items():
    exp = EXPECT_WRITES.get(k)
    ok = exp is None or v == exp
    print(("  ✅ " if ok else "  ❌ ") + f"{k}: {v}" + ("" if ok else f" (expected {exp})"))
    hb(S, ("PASS " if ok else "FAIL ") + f"{k}: {v}")
hb(S, f".bak {BAK_BYTES / 1e6:,.1f} MB, export {TIMERS['export .bak']:,.0f} s; write tests rolled back")
send_back(S)

---
## 8 · Restore as Start Lab will

The build copy is dropped and the `.bak` is imported as `CymbalGoalScouting` with
`gcloud sql import bak`, from the staging bucket: the same call a Start Lab setup script makes, so its
time is the number the 20-minute budget needs. Row counts are then checked against the load.

In [ ]:
S = "8 · Restore"
master = connect()
drop_database(master, DB_BUILD)
with timed("import .bak (Start Lab step)"):
    op = sh(f"gcloud sql import bak {SQL_INSTANCE} {BAK_STAGING} --database={DB_FINAL} --async --quiet --format='value(name)'")
    info = wait_sql_op(op, "import")

final = connect(DB_FINAL)
RESTORED = {t: int(scalar(final, f"SELECT COUNT_BIG(*) FROM dbo.{t}")) for t in LOAD_ORDER}
mismatch = {t: (RESTORED[t], LOAD_STATS[t]["rows"]) for t in LOAD_ORDER if RESTORED[t] != LOAD_STATS[t]["rows"]}
FINAL_COLLATION = scalar(master, "SELECT CONVERT(NVARCHAR(128), DATABASEPROPERTYEX(%s, 'Collation'))", DB_FINAL)
RECOVERY = scalar(master, "SELECT recovery_model_desc FROM sys.databases WHERE name = %s", DB_FINAL)
OWNER = scalar(master, "SELECT SUSER_SNAME(owner_sid) FROM sys.databases WHERE name = %s", DB_FINAL)
print("rows after restore:", ", ".join(f"{t} {n:,}" for t, n in RESTORED.items()))
print(f"collation {FINAL_COLLATION}, recovery {RECOVERY}, owner {OWNER}")
print("✅ every count matches the load" if not mismatch else f"❌ mismatches: {mismatch}")
hb(S, f"import .bak: {TIMERS['import .bak (Start Lab step)']:,.0f} s for {BAK_BYTES / 1e6:,.1f} MB")
hb(S, "row counts after restore: " + ("all match" if not mismatch else f"MISMATCH {mismatch}"))
hb(S, f"collation {FINAL_COLLATION}, recovery {RECOVERY}, owner {OWNER}")
send_back(S)

---
## 9 · Change data capture

`03-enable-cdc.sql`, exactly as Start Lab would run it: CDC on the database, snapshot isolation on,
then a capture instance per table. A DMS continuous job reads these change tables.

In [ ]:
S = "9 · CDC"
with timed("enable CDC"):
    _, FAILED_CDC = run_batches(master, split_batches(SCRIPTS["03-enable-cdc.sql"]), "enable CDC")
final = connect(DB_FINAL)
cur = final.cursor()
cur.execute("SELECT name, is_tracked_by_cdc FROM sys.tables WHERE schema_id = SCHEMA_ID('dbo') ORDER BY name")
TRACKED = {r[0]: bool(r[1]) for r in cur.fetchall()}
DB_CDC = scalar(master, "SELECT is_cdc_enabled FROM sys.databases WHERE name = %s", DB_FINAL)
SNAPSHOT = scalar(master, "SELECT snapshot_isolation_state_desc FROM sys.databases WHERE name = %s", DB_FINAL)
try:
    cur.execute("EXEC sys.sp_cdc_help_jobs")
    JOBS = [f"{r[1]}" for r in cur.fetchall()]
except Exception as e:  # noqa: BLE001
    JOBS = [f"(sp_cdc_help_jobs: {type(e).__name__}: {str(e)[:120]})"]
print(f"database CDC {bool(DB_CDC)}, snapshot isolation {SNAPSHOT}")
print("tracked:", ", ".join(f"{k}={'yes' if v else 'NO'}" for k, v in TRACKED.items()))
print("CDC jobs:", ", ".join(JOBS))
hb(S, f"enable CDC: {TIMERS['enable CDC']:,.1f} s; failed batches {len(FAILED_CDC)}")
for f in FAILED_CDC:
    hb(S, "  ❌ " + f)
hb(S, f"database CDC {bool(DB_CDC)}; snapshot isolation {SNAPSHOT}; tables tracked "
      f"{sum(TRACKED.values())}/{len(TRACKED)}; jobs: {', '.join(JOBS)}")
send_back(S)

---
## 10 · Publish

To `gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop/sqlserver/`:

| Object | What it is |
| :-- | :-- |
| `CymbalGoalScouting.bak` | the full backup, CDC **not** enabled (Start Lab runs `03-enable-cdc.sql` after the import) |
| `01-schema.sql` | the schema **as it ran** (any widened `NVARCHAR` and the `game_event_id` type applied) |
| `02-programmability.sql`, `03-enable-cdc.sql` | as fetched |
| `smoke-sqlserver.json` | section 6's calls and SQL Server's answers, for the conversion comparison in step 3 |
| `manifest.json` | row counts, column types, objects, script hashes, `.bak` size and checksum, timings |

In [ ]:
S = "10 · Publish"
pub = WORK / "publish"
pub.mkdir(exist_ok=True)
(pub / "01-schema.sql").write_text(SCHEMA_SQL, encoding="utf-8")
(pub / "02-programmability.sql").write_text(SCRIPTS["02-programmability.sql"], encoding="utf-8")
(pub / "03-enable-cdc.sql").write_text(SCRIPTS["03-enable-cdc.sql"], encoding="utf-8")
(pub / "smoke-sqlserver.json").write_text((WORK / "smoke-sqlserver.json").read_text())

MANIFEST = {
    "built_at_utc": RUN_ID, "builder": "pipeline/05-sqlserver-package.ipynb", "project": PROJECT,
    "sqlserver": {"version": VERSION, "instance_tier": inst["settings"]["tier"],
                  "database_version": inst["databaseVersion"], "collation": FINAL_COLLATION},
    "database": DB_FINAL, "season_from": SEASON_FROM, "source": GCS_READ,
    "source_snapshot": MF.get("snapshot", {}),
    "tables": {t: {"rows": LOAD_STATS[t]["rows"],
                   "columns": [[c["name"], c["type"] + (" IDENTITY" if c["identity"] else "")] for c in SCHEMA[t]]}
               for t in LOAD_ORDER},
    "objects": [[k, n] for k, n in OBJECTS],
    "nvarchar_widened": WIDENED,
    "scripts_sha256": {n: hashlib.sha256((pub / n).read_bytes()).hexdigest() for n in SCRIPT_NAMES},
    "bak": {"uri": f"{PKG_PREFIX}/{BAK_NAME}", "bytes": BAK_BYTES,
            "crc32c": BAK_META.get("crc32c_hash"), "md5": BAK_META.get("md5_hash"), "cdc_enabled_in_bak": False},
    "deliberate_trap": "usp_search_players: LIKE under a case-insensitive collation; PostgreSQL LIKE is case-sensitive",
    "timings_seconds": {k: round(v, 1) for k, v in TIMERS.items()},
}
(pub / "manifest.json").write_text(json.dumps(MANIFEST, indent=2))

with timed("publish"):
    r1 = gcs("cp", BAK_STAGING, f"{PKG_PREFIX}/{BAK_NAME}")
    r2 = gcs("cp", *[pub / n for n in ["01-schema.sql", "02-programmability.sql", "03-enable-cdc.sql",
                                        "smoke-sqlserver.json", "manifest.json"]], f"{PKG_PREFIX}/")
for r in (r1, r2):
    if r.returncode != 0:
        print("❌", r.stderr.strip()[:400])
listing = gcs("ls", "-l", f"{PKG_PREFIX}/")
print(listing.stdout)
hb(S, f"published to {PKG_PREFIX}/: " + ("OK" if r1.returncode == r2.returncode == 0 else "FAILED (see cell output)"))
hb(S, f".bak {BAK_BYTES / 1e6:,.1f} MB crc32c {BAK_META.get('crc32c_hash')}")
send_back(S)

---
## 11 · Handback

Everything from the **📋 SEND BACK** blocks, plus the timers and the bucket audit. Copy the whole output
of this cell and paste it back. It's also saved to the workshop bucket under `module1-checks/`.

In [ ]:
lines = ["=" * 78,
         f"MODULE 1 CHECKS: STEP 2 SQL SERVER PACKAGE HANDBACK  (05-sqlserver-package.ipynb)  run {RUN_ID}",
         "=" * 78]
for sec, ls_ in HB.items():
    lines += ["", f"## {sec}", *ls_]
lines += ["", "## Timers"]
lines += [f"{k}: {v:,.1f} s" for k, v in TIMERS.items()]
writes = [c for c in GCS_LOG if c.split()[2] in ("cp", "mv", "rm")]
bad = [c for c in writes if (GCS_READ + "/") in c.split()[-1]]
lines += ["", "## Bucket audit",
          f"gcloud storage calls {len(GCS_LOG)}, writes {len(writes)}, writes under the read-only prefix {len(bad)}",
          "=" * 78]
text = "\n".join(lines)
print(text)
(WORK / "step2-handback.txt").write_text(text)
r = gcs("cp", WORK / "step2-handback.txt", f"{RESULTS_PREFIX}/step2-handback-{RUN_ID}.txt")
print(f"\n[saved to {RESULTS_PREFIX}/]" if r.returncode == 0 else f"\n[not saved: {r.stderr.strip()[:200]}]")
assert not bad, f"a write reached the read-only prefix: {bad}"